In [ ]:
# Load packages
import os
import hail as hl
import pandas as pd 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Loading BokehJS ...

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261021 DATAPROC_CLUSTER
JupyterLab_Spark_cluster_for_AoU_Spark_20261021_2 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_

In [3]:
# Run this cell once
hl.init(
    gcs_requester_pays_configuration=os.environ['GOOGLE_CLOUD_PROJECT']
)

Running on Apache Spark version 3.5.3
SparkUI available at http://jupyterlabsparkclusterforaouspark202610212-m.us-west2-c.c.wb-fast-radish-6415.internal:42147
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.135-034ef3e08116
LOGGING: writing to /home/dataproc/workspace/rw-migration-aou-rw-9f879fc4/notebooks/hail/hail-20261022-1912-0.2.135-034ef3e08116.log
2026-10-22 19:13:12.980 Hail: INFO: Found 218625 samples in fam file.
2026-10-22 19:13:12.981 Hail: INFO: Found 4910 variants in bim file.
2026-10-22 19:13:19.547 Hail: INFO: Found 218625 samples in fam file.
2026-10-22 19:13:19.547 Hail: INFO: Found 4910 variants in bim file.
2026-10-22 19:16:41.026 Hail: INFO: Found 218625 samples in fam file.
2026-10-22 19:16:41.026 Hail: INFO: Found 4910 variants in bim file.
2026-10-22 19:20:02.745 Hail: INFO: Found 218625 samples in fam file.
2026-10-22 19:20:02.746 Hail: INFO: Found 4910 variants in bim file.
2026-10-22 19:20:22.207 Hail:

In [4]:
import os
import subprocess
import pandas as pd

# Store all sample|variant exclusion pairs
remove_pairs = set()

bucket = os.environ["WORKSPACE_BUCKET"]

# GCS source and local destination
gcs_path = f"{bucket}/data/chip/remove.txt"
local_path = f"{os.getcwd()}/remove.txt"

# Download from Google Cloud Storage
subprocess.run(
    ["gcloud", "storage", "cp", gcs_path, local_path],
    check=True
)

# Read the downloaded file
remove_df = pd.read_csv(
    local_path,
    sep="\t",
    dtype=str
)

# Create sample|variant pairs
remove_pairs = set(
    remove_df["sample"] + "|" + remove_df["variant"]
)

print("Total unique exclusions across chromosomes:", len(remove_pairs))

Copying gs://rw-migration-aou-rw-9f879fc4/data/chip/remove.txt to file:///home/dataproc/workspace/rw-migration-aou-rw-9f879fc4/notebooks/hail/remove.txt
  
.............

Average throughput: 60.0MiB/s


Total unique exclusions across chromosomes: 972366


In [5]:


for chrom in [4, 15, 17]:

    prefix = (
        f"{os.environ['WORKSPACE_BUCKET']}/data/chip/"
        f"plink/EUR_passed_chr{chrom}"
    )

    # 1. Import PLINK
    mt = hl.import_plink(
        bed=f"{prefix}.bed",
        bim=f"{prefix}.bim",
        fam=f"{prefix}.fam"
    )

    # 2. Identify matching sample-variant calls
    remove_pairs_hl = hl.literal(remove_pairs)

    mt = mt.annotate_entries(
        _target=remove_pairs_hl.contains(mt.s + "|" + mt.rsid)
    )

    # 3. Mask matching genotypes
    print("Matching calls:", mt.aggregate_entries(
        hl.agg.count_where(mt._target)
    ))

    mt = mt.annotate_entries(
        GT=hl.if_else(mt._target, hl.missing(hl.tcall), mt.GT)
    )

    remaining = mt.aggregate_entries(
        hl.agg.count_where(mt._target & hl.is_defined(mt.GT))
    )
    print("Target calls still defined (expected 0):", remaining)
    assert remaining == 0

    # 5. Export PLINK
    hl.export_plink(
        mt,
        f"{os.environ['WORKSPACE_BUCKET']}/data/chip/"
        f"plink/EUR_ab0.35_dp20_chr{chrom}"
    )

Matching calls: 6877


Target calls still defined (expected 0): 0


Matching calls: 1262


Target calls still defined (expected 0): 0


Matching calls: 3073


Target calls still defined (expected 0): 0


In [6]:
!gcloud storage ls {os.environ['WORKSPACE_BUCKET']}/data/chip/plink/

gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr15.bed
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr15.bim
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr15.fam
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr17.bed
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr17.bim
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr17.fam
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr2.bed
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr2.bim
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr2.fam
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr4.bed
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr4.bim
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_ab0.35_dp20_chr4.fam
gs://rw-migration-aou-rw-9f879fc4/data/chip/plink/EUR_passed_chr15.bed
gs://rw-migration-aou-r